# Luhya ASR comparison: Whisper Small vs Wav2Vec2-BERT 2.0

This Kaggle notebook runs both model families sequentially on the same immutable dataset revision and speaker-disjoint train/validation/test protocol. It generates exact WER/CER artifacts, a paired comparison report, current Hugging Face model cards, optional private Hub uploads, and a bounded GitHub results commit.

Before running, select **GPU T4 x2**, enable **Internet**, and add Kaggle secrets named `HF_TOKEN` and (only if GitHub publishing is enabled) `GITHUB_TOKEN`. W&B is not used. The Hub uploads default to **private** because the dataset redistribution terms still need to be confirmed.

In [ ]:
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch; "
            "print('CUDA devices:', torch.cuda.device_count()); "
            "assert torch.cuda.device_count() == 2, 'Select GPU T4 x2 in Kaggle settings'"
        ),
    ],
    check=True,
)

In [ ]:
repo_dir = Path("/kaggle/working/luhya-asr")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/sam4rano/luhya-asr.git", str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
code_revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print("Repository:", Path.cwd())
print("Training code revision:", code_revision)

In [ ]:
# Keep Kaggle's CUDA-compatible PyTorch; install only the pinned ASR stack.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
    check=True,
)

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert os.environ["HF_TOKEN"], "HF_TOKEN is missing from Kaggle Secrets"

api = HfApi(token=os.environ["HF_TOKEN"])
account = api.whoami(token=os.environ["HF_TOKEN"])
dataset_id = "Digital-Divide-Data/Luhya-ASR-Data-subset-50h"
dataset_info = api.dataset_info(dataset_id, token=os.environ["HF_TOKEN"])
dataset_revision = dataset_info.sha
print("Authenticated as:", account["name"])
print("Dataset:", dataset_info.id)
print("Pinned dataset revision:", dataset_revision)

## Run controls

Both models are enabled by default. If a Kaggle session stops, rerun the notebook with the completed model's `RUN_*` flag set to `False`; the other model resumes from its latest retained checkpoint. Keep both `SMOKE_TEST_*` flags enabled on the first attempt.

In [ ]:
RUN_WHISPER = True
RUN_WAV2VEC2_BERT = True
SMOKE_TEST_WHISPER = True
SMOKE_TEST_WAV2VEC2_BERT = True
PUBLISH_TO_HUGGING_FACE = True
HF_PRIVATE = True
PUSH_RESULTS_TO_GITHUB = True
PUSH_PREDICTIONS_TO_GITHUB = False  # safer default: prediction rows include speaker metadata

HF_REPO_NAMES = {
    "whisper": "luhya-whisper-small-40h",
    "wav2vec2_bert": "luhya-wav2vec2-bert-40h",
}

output_root = Path("/kaggle/working/luhya-asr-output")
whisper_dir = output_root / "whisper-small-luhya"
wav2vec_dir = output_root / "wav2vec2-bert-luhya"
comparison_dir = output_root / "comparison-40h"
output_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# Resolve `main` once, then give both trainers the exact same dataset commit.
import yaml

runtime_config_dir = Path("/kaggle/working/luhya-runtime-configs")
runtime_config_dir.mkdir(parents=True, exist_ok=True)

def pinned_config(source_name, destination_name):
    with (repo_dir / "config_files" / source_name).open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    config["dataset_revision"] = dataset_revision
    config["code_revision"] = code_revision
    destination = runtime_config_dir / destination_name
    with destination.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(config, handle, sort_keys=False)
    return destination

whisper_config = pinned_config(
    "ASR_train_config_whisper_small_kaggle.yaml",
    "whisper.yaml",
)
wav2vec_config = pinned_config(
    "ASR_train_config_wav2vec2_bert_kaggle.yaml",
    "wav2vec2_bert.yaml",
)
print(whisper_config)
print(wav2vec_config)

In [ ]:
accelerate_prefix = [
    "accelerate", "launch",
    "--multi_gpu",
    "--num_processes", "2",
    "--mixed_precision", "fp16",
    "--num_cpu_threads_per_process", "2",
]
whisper_launch = accelerate_prefix + [
    "scripts/train_whisper.py", "--config", str(whisper_config)
]
wav2vec_launch = accelerate_prefix + [
    "scripts/train_model.py", "--config", str(wav2vec_config)
]
print("Whisper command:", " ".join(whisper_launch))
print("Wav2Vec2-BERT command:", " ".join(wav2vec_launch))

## Resource-safety smoke tests

Each smoke test uses deterministic small subsets and two optimizer steps in a separate output directory. The full jobs start only after their smoke test succeeds.

In [ ]:
if RUN_WHISPER and SMOKE_TEST_WHISPER:
    subprocess.run(whisper_launch + ["--smoke_test"], check=True)
if RUN_WAV2VEC2_BERT and SMOKE_TEST_WAV2VEC2_BERT:
    subprocess.run(wav2vec_launch + ["--smoke_test"], check=True)
print("Enabled smoke tests passed.")

In [ ]:
# Reclaim smoke-test checkpoints before the full sequential runs.
smoke_dirs = [
    whisper_dir / "smoke-test",
    output_root / "wav2vec2-bert-luhya-smoke-test",
]
for smoke_dir in smoke_dirs:
    if smoke_dir.exists():
        shutil.rmtree(smoke_dir)
        print("Removed:", smoke_dir)

## Full training and held-out evaluation

The models run sequentially so they never compete for GPU memory. `latest` resumes an interrupted run when a checkpoint exists. Each trainer writes validation/test metrics, exact test prediction rows, a split manifest, and a reloadable `final-model/`.

In [ ]:
if RUN_WHISPER:
    subprocess.run(
        whisper_launch + ["--resume_from_checkpoint", "latest"],
        check=True,
    )
else:
    print("Whisper training skipped; existing artifacts will be used.")

In [ ]:
if RUN_WAV2VEC2_BERT:
    subprocess.run(
        wav2vec_launch + ["--resume_from_checkpoint", "latest"],
        check=True,
    )
else:
    print("Wav2Vec2-BERT training skipped; existing artifacts will be used.")

## Comparison gate and shareable report

Report generation stops if the dataset revisions, split manifests, speaker-overlap audit, prediction row order, normalized references, or exact recomputed test metrics do not match. This prevents an invalid comparison from being published.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/build_comparison_report.py",
        "--whisper-dir", str(whisper_dir),
        "--wav2vec-dir", str(wav2vec_dir),
        "--output-dir", str(comparison_dir),
    ],
    check=True,
)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

comparison = pd.read_csv(comparison_dir / "comparison_summary.csv")
display(comparison.style.format({
    "wer_percent": "{:.2f}%",
    "cer_percent": "{:.2f}%",
}))
display(Markdown((comparison_dir / "Luhya_ASR_Experiment_Report.md").read_text()))

ax = (
    comparison.pivot(index="model", columns="split", values="wer_percent")
    .plot.bar(figsize=(8, 4), ylabel="WER (%)", rot=0, title="Luhya ASR comparison")
)
figure = ax.get_figure()
figure.tight_layout()
figure.savefig(comparison_dir / "comparison_wer.png", dpi=160, bbox_inches="tight")

## Publish current model artifacts to Hugging Face

The generated model cards contain the measured metrics and immutable data/code revisions. Repositories default to private; change `HF_PRIVATE` only after confirming consent, dataset licensing, and weight redistribution terms.

In [ ]:
if PUBLISH_TO_HUGGING_FACE:
    namespace = account["name"]
    upload_targets = {
        "whisper": whisper_dir / "final-model",
        "wav2vec2_bert": wav2vec_dir / "final-model",
    }
    for label, folder in upload_targets.items():
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing final model: {folder}")
        repo_id = f"{namespace}/{HF_REPO_NAMES[label]}"
        api.create_repo(
            repo_id=repo_id,
            repo_type="model",
            private=HF_PRIVATE,
            exist_ok=True,
            token=os.environ["HF_TOKEN"],
        )
        commit = api.upload_folder(
            repo_id=repo_id,
            repo_type="model",
            folder_path=folder,
            commit_message=f"Upload aligned 40h Luhya ASR run ({code_revision[:12]})",
            token=os.environ["HF_TOKEN"],
        )
        print(label, commit.commit_url)
else:
    print("Hugging Face publishing disabled.")

## Publish bounded results to GitHub

Model weights and checkpoints are never committed to GitHub. By default, speaker-level prediction files are also excluded; the report, manifests, aggregate metrics, plot, and model-specific summaries are committed.

In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    assert github_token, "GITHUB_TOKEN is missing from Kaggle Secrets"

    askpass = Path("/kaggle/temp/github-askpass.sh")
    askpass.parent.mkdir(parents=True, exist_ok=True)
    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' "x-access-token" ;;
  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;
esac
"""
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
    git_env = os.environ.copy()
    git_env.update({
        "GITHUB_TOKEN": github_token,
        "GIT_ASKPASS": str(askpass),
        "GIT_ASKPASS_REQUIRE": "force",
        "GIT_TERMINAL_PROMPT": "0",
    })

    def git(*arguments, check=True):
        return subprocess.run(
            ["git", "-C", str(repo_dir), *arguments],
            env=git_env,
            check=check,
            text=True,
        )

    try:
        git("config", "user.name", "sam4rano")
        git("config", "user.email", "sam4rano@users.noreply.github.com")
        git("pull", "--rebase", "origin", "main")

        destinations = {
            whisper_dir: repo_dir / "results" / "whisper-small-40h-aligned",
            wav2vec_dir: repo_dir / "results" / "wav2vec2-bert-40h-aligned",
            comparison_dir: repo_dir / "results" / "comparison-40h",
        }
        base_files = ["evaluation_summary.json", "metrics_summary.csv"]
        comparison_files = [
            "Luhya_ASR_Experiment_Report.md",
            "comparison_manifest.json",
            "comparison_summary.csv",
            "comparison_wer.png",
            "dialect_metrics.csv",
        ]
        if PUSH_PREDICTIONS_TO_GITHUB:
            base_files.append("test_predictions.csv")
            comparison_files.extend(["paired_test_predictions.csv", "speaker_metrics.csv"])

        for source_dir, destination_dir in destinations.items():
            destination_dir.mkdir(parents=True, exist_ok=True)
            selected = comparison_files if source_dir == comparison_dir else base_files
            for filename in selected:
                source = source_dir / filename
                if source.is_file():
                    shutil.copy2(source, destination_dir / filename)
                    print("Prepared:", destination_dir / filename)

        git("add", "results/whisper-small-40h-aligned")
        git("add", "results/wav2vec2-bert-40h-aligned")
        git("add", "results/comparison-40h")
        changes = git("diff", "--cached", "--quiet", check=False)
        if changes.returncode == 0:
            print("No new result changes to commit.")
        else:
            git("commit", "-m", "Add aligned Luhya ASR comparison results")
            git("push", "origin", "main")
            print("Comparison results pushed successfully.")
    finally:
        git_env.pop("GITHUB_TOKEN", None)
        github_token = None
        askpass.unlink(missing_ok=True)
else:
    print("GitHub result publishing disabled.")

## Recovery commands

If training completed but report/export failed, do **not** retrain. Run the relevant evaluation-only command, then rerun the comparison and publishing cells:

```python
subprocess.run(whisper_launch + ["--evaluation_only"], check=True)
subprocess.run(wav2vec_launch + ["--evaluation_only"], check=True)
```